# Exploratory Data Analysis — What Predicts Campaign Response?

Bivariate checks on the cleaned dataset (`CP_data_cleaning.ipynb` output): response rates against demographics, engagement,
and CRM history — with significance tests.
Output feeds into `cp_modeling.ipynb`.

## Setup

In [ ]:
import yaml
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency

In [ ]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

data = pd.read_csv(config['data']['clean']['file5'], quotechar='"', sep=";")
data.shape

## Correlation overview

In [ ]:
data.select_dtypes('number').corr()

## Age vs. response

`age` was derived from birth year during cleaning. 

In [ ]:
data.groupby('age')['response'].mean()

In [ ]:
conv_success = data[data['response'] == 1].age
conv_failure = data[data['response'] == 0].age
t_stat, p_value = ttest_ind(conv_success, conv_failure, equal_var=False, alternative="two-sided")
print(f"t = {t_stat:.4f}, p = {p_value:.4e}")

## Age vs. income

In [ ]:
data[['age', 'income']].corr()

No meaningful correlation between age and income in this dataset — worth knowing before treating either as a proxy for the other.

## Income vs. response

In [ ]:
sns.boxplot(data=data, x='response', y='income')
plt.title("Income by campaign response")
plt.show()

In [ ]:
conv_success = data[data['response'] == 1].income
conv_failure = data[data['response'] == 0].income
t_stat, p_value = ttest_ind(conv_success, conv_failure, equal_var=False, alternative="two-sided")
print(f"t = {t_stat:.4f}, p = {p_value:.4e}")

Responders have significantly higher income than non-responders (Welch's t-test, unequal variances assumed since group sizes and spreads differ).

## Education vs. response

In [ ]:
data.groupby('education')['response'].mean()

In [ ]:
table = pd.crosstab(data['education'], data['response'])
chi2, p, dof, expected = chi2_contingency(table)
print(f"chi2 = {chi2:.4f}, p = {p:.4e}")

## Marital status vs. response

In [ ]:
data.groupby('marital_status')['response'].mean()

In [ ]:
table = pd.crosstab(data['marital_status'], data['response'])
chi2, p, dof, expected = chi2_contingency(table)
print(f"chi2 = {chi2:.4f}, p = {p:.4e}")

## Complaints vs. response

In [ ]:
data.groupby('complain')['response'].mean()

## Children vs. response

In [ ]:
data.groupby('num_children')['response'].mean()

In [ ]:
conv_success = data[data['response'] == 1].num_children
conv_failure = data[data['response'] == 0].num_children
t_stat, p_value = ttest_ind(conv_success, conv_failure, equal_var=False, alternative="two-sided")
print(f"t = {t_stat:.4f}, p = {p_value:.4e}")

In [ ]:
# Has_child is boolean (True/False), which ttest_ind treats as 1/0 — same test,
# coarser lens than the exact child count above.
conv_success = data[data['response'] == 1].has_child
conv_failure = data[data['response'] == 0].has_child
t_stat, p_value = ttest_ind(conv_success, conv_failure, equal_var=False, alternative="two-sided")
print(f"t = {t_stat:.4f}, p = {p_value:.4e}")

## Past campaign success vs. response

In [ ]:
data.groupby('num_camp_success')['response'].mean()

In [ ]:
conv_success = data[data['response'] == 1].num_camp_success
conv_failure = data[data['response'] == 0].num_camp_success
t_stat, p_value = ttest_ind(conv_success, conv_failure, equal_var=False, alternative="two-sided")
print(f"t = {t_stat:.4f}, p = {p_value:.4e}")

## Export

In [ ]:
data.columns

In [ ]:
data.to_csv("../data/clean/CP_ready_to_model.csv", index=False, encoding="utf-8", sep=";")

## Summary

Age and income are not correlated. Campaign response is higher for higher
income, higher education, and customers who live alone with no children.
Prior campaign success is the strongest signal of all — people who've
responded before are far more likely to respond again, which is the basis
for including `num_camp_success` as a core feature in the modeling stage.